## IMPLEMENTACION CON PYCARET

In [1]:
# 1. IMPORTA PRIMERO PYCARET 
import os
import sys
import hydra  # 
from hydra import initialize, compose
import pandas as pd
import numpy as np
from pycaret.regression import *

In [2]:
import pandas as pd
from hydra import initialize, compose
import hydra

# 1. Limpiar cualquier instancia previa de Hydra de forma segura
if hydra.core.global_hydra.GlobalHydra.instance().is_initialized():
    hydra.core.global_hydra.GlobalHydra.instance().clear()

# 2. Inicializar Hydra 
with initialize(version_base=None, config_path="../config"):
    # 3. Cargar la configuración 
    cfg = compose(config_name="config")

# 4. Extraer las columnas desde tu archivo YAML de Hydra
text_features = list(cfg.features.text)
numeric_features = list(cfg.features.numeric)
binary_features = list(cfg.features.binary)
target = cfg.target

# Combinamos todas las columnas necesarias para no leer el archivo completo en vano
cols_to_read = text_features + numeric_features + binary_features + [target]

# 5. Leer el DataFrame filtrando solo las columnas del YAML
df_raw = pd.read_csv(
    cfg.data.path,
    usecols=cols_to_read
)


df_raw.head()




,host_is_superhost,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bedrooms,beds,price,guests_included,minimum_nights,number_of_reviews,review_scores_rating,instant_bookable
0,t,Copacabana,Condominium,Entire home/apt,5,1.0,2.0,2.0,$332.00,2,4,243,93.0,t
1,f,Copacabana,Apartment,Entire home/apt,2,1.0,1.0,2.0,$160.00,2,7,235,94.0,f
2,t,Ipanema,Apartment,Entire home/apt,3,1.0,1.0,2.0,$273.00,2,2,271,96.0,t
3,t,Ipanema,Apartment,Entire home/apt,3,1.5,1.0,2.0,$378.00,2,2,169,94.0,f
4,t,Copacabana,Loft,Entire home/apt,2,1.0,1.0,1.0,$130.00,2,3,316,98.0,f


In [3]:
# 1. Añadir la raíz del proyecto al path de Python
sys.path.append(os.path.abspath(".."))

# 2. importar  función 
from src.plots.plots import generar_diccionario

generar_diccionario(df_raw)


-------TABLA CON TIPO DE VARIABLE, VALORES UNICOS, % DE NULOS Y MODA-------



,Variable,Tipo pandas,Cantidad de valores únicos,% Valores faltantes,Valor más frecuente (Moda)
0,host_is_superhost,object,2,0.06,f
1,neighbourhood_cleansed,object,153,0.00,Copacabana
2,property_type,object,37,0.00,Apartment
3,room_type,object,4,0.00,Entire home/apt
4,accommodates,int64,22,0.00,4
5,bathrooms,float64,33,0.16,1.0
6,bedrooms,float64,17,0.12,1.0
7,beds,float64,38,0.14,1.0
8,price,object,856,0.00,$202.00
9,guests_included,int64,21,0.00,1


In [4]:
import sys
import os

# 1. Subimos un nivel ("..") para apuntar a la raíz del proyecto y lo añadimos al camino de Python
raiz_proyecto = os.path.abspath(os.path.join(".."))
if raiz_proyecto not in sys.path:
    sys.path.append(raiz_proyecto)

# 2. 
from src.processing.clean_target import clean_raw_data

# 3. Limpiamos el target
df_cleaned = clean_raw_data(df_raw, columna_target=target)
print(" ¡Datos limpiados con éxito!")


# 4. Forzar las columnas que deben ser numéricas a tipo float
print(" Purificando variables numéricas leídas como object...")
for col in numeric_features:
    if col in df_cleaned.columns:
        # errors='coerce' convierte cualquier texto basura o espacio en un NaN numérico limpio
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').astype(float)

print(" Variables de tipo object convertidas a float correctamente.")

 ¡Datos limpiados con éxito!
 Purificando variables numéricas leídas como object...
 Variables de tipo object convertidas a float correctamente.


In [5]:
from pycaret.regression import *
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# ----------------------------------------------------------------------
#  Filtrar solo las columnas que SÍ existen en el CSV
# ----------------------------------------------------------------------
todas_las_categoricas = text_features + binary_features

# Asegurar de que PyCaret y Sklearn solo busquen columnas reales
categoricas_reales = [col for col in todas_las_categoricas if col in df_cleaned.columns]
numericas_reales = [col for col in numeric_features if col in df_cleaned.columns]


columnas_faltantes = set(todas_las_categoricas + numeric_features) - set(df_cleaned.columns)
if columnas_faltantes:
    print(f"¡Atención! Estas columnas están en tu YAML pero NO existen en archivo CSV: {columnas_faltantes}")
else:
    print("Todas las columnas del YAML coinciden con el archivo CSV.")

Todas las columnas del YAML coinciden con el archivo CSV.


In [6]:
import pandas as pd
from src.processing.clean_target import clean_raw_data

# 1.limpieza (Target)
df_cleaned = clean_raw_data(df_raw, columna_target=target)

# 2. Forzar las columnas que deben ser numéricas a tipo float
for col in numeric_features:
    if col in df_cleaned.columns:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').astype(float)

# 3. FILTRO DE ALTA CARDINALIDAD DIRECTO EN PANDAS
print(" Agrupando categorías menores al 0.6% en 'others'...")
threshold = 0.006
other_label = 'others'

# filtrar las variables de texto (barrios, tipos de propiedad)
for col in text_features:
    if col in df_cleaned.columns:
        # Calcular la frecuencia relativa de cada valor
        frecuencias = df_cleaned[col].value_counts(normalize=True)
        # Identificamos los valores que SÍ cumplen con el mínimo del 0.6%
        valores_frecuentes = frecuencias[frecuencias >= threshold].index.tolist()
        
        # Si el valor no es frecuente, lo mandamos a 'others'
        df_cleaned[col] = df_cleaned[col].apply(
            lambda x: x if (pd.isna(x) or x in valores_frecuentes) else other_label
        )

print(" ¡Datos reducidos y purificados con éxito!")


 Agrupando categorías menores al 0.6% en 'others'...
 ¡Datos reducidos y purificados con éxito!


In [7]:
from pycaret.regression import *
from sklearn.pipeline import Pipeline

# Importar las clases personalizadas desde el archivo
from src.processing.transformers import (
    MissingIndicatorTargeted, 
    CleanBinaryFeatures, 
    CleanAndGroupTextFeatures
)

# 1. Listas extraídas desde Hydra
todas_las_categoricas = [col for col in (text_features + binary_features) if col in df_cleaned.columns]
numericas_reales = [col for col in numeric_features if col in df_cleaned.columns]
columnas_binarias = [col for col in binary_features if col in df_cleaned.columns]
columnas_texto = [col for col in text_features if col in df_cleaned.columns]

# 2. Implementar Pipeline
pipeline_personalizado = Pipeline([
    ('indicador_nulos_20', MissingIndicatorTargeted(threshold=0.20)),
    ('limpieza_binarias', CleanBinaryFeatures(binary_columns=columnas_binarias)),
    ('limpieza_y_agrupacion_texto', CleanAndGroupTextFeatures(
        text_columns=columnas_texto, 
        threshold=0.006, 
        other_label='otras'
    ))
])

# 3. Aplicamos log1p directamente a la columna precio
df_cleaned[target] = np.log1p(df_cleaned[target])

# 4. Setup para PyCaret
reg_setup = setup(
    data=df_cleaned, 
    target=target,
    numeric_features=numericas_reales,
    categorical_features=todas_las_categoricas,
    custom_pipeline=pipeline_personalizado, 
    transform_target=False, 
    numeric_imputation='mean',         
    categorical_imputation='mode',     
    session_id=42,
    preprocess=True,                    
    verbose=True
)





,Description,Value
0,Session id,42
1,Target,price
2,Target type,Regression
3,Original data shape,"(33708, 14)"
4,Transformed data shape,"(33708, 46)"
5,Transformed train set shape,"(23595, 46)"
6,Transformed test set shape,"(10113, 46)"
7,Numeric features,8
8,Categorical features,5
9,Rows with missing values,45.8%


In [8]:
# Comparamos todos los modelos de regresión disponibles
modelo_ganador = compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.4874,0.4270,0.6532,0.6027,0.0946,0.0855,0.7000
gbr,Gradient Boosting Regressor,0.5072,0.4550,0.6743,0.5767,0.0978,0.0891,1.2320
rf,Random Forest Regressor,0.5126,0.4780,0.6911,0.5554,0.1003,0.0899,5.3350
et,Extra Trees Regressor,0.5291,0.5150,0.7174,0.5209,0.1041,0.0927,6.3570
knn,K Neighbors Regressor,0.5651,0.5613,0.7489,0.4778,0.1095,0.1004,0.4980
ada,AdaBoost Regressor,0.6217,0.6288,0.7925,0.4150,0.1176,0.1143,0.6490
ridge,Ridge Regression,0.5886,0.6447,0.7978,0.3989,0.1111,0.1032,0.1800
br,Bayesian Ridge,0.5887,0.6449,0.7979,0.3988,0.1111,0.1032,0.2310
lr,Linear Regression,0.5890,0.6451,0.7980,0.3986,0.1111,0.1032,0.8110
huber,Huber Regressor,0.6183,0.7533,0.8584,0.2972,0.1216,0.1076,0.6840


In [9]:
# Entrenamos el LightGBM con absolutamente todos los datos
modelo_final_lgb = finalize_model(modelo_ganador)

In [10]:
import os
import numpy as np
from sklearn.compose import TransformedTargetRegressor
from pycaret.regression import save_model, get_config

# 1. definir X_train y y_train 
X_train_pycaret = get_config('X_train')
y_train_pycaret = get_config('y_train')

# 2. revertir el logaritmo de forma nativa
modelo_empaquetado_api = TransformedTargetRegressor(
    regressor=modelo_final_lgb,  # <--- Asegúrate de que tu modelo final se llame así
    func=np.log1p,         
    inverse_func=np.expm1  
)

# 3. Entrenamor el modelo empaquetado usando los datos extraídos de PyCaret
modelo_empaquetado_api.fit(X_train_pycaret, y_train_pycaret)

# 4. Crear la carpeta 'app' si no existe
os.makedirs("app", exist_ok=True)



[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002928 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 455
[LightGBM] [Info] Number of data points in the train set: 23595, number of used features: 45
[LightGBM] [Info] Start training from score 1.902945


In [ ]:
import os
from pycaret.regression import save_model

# 1. Obtenemos la ruta actual 
ruta_actual = os.getcwd()

# 2. Subir un nivel a la raíz (Proyecto_3) y apuntamos a la carpeta 'app' 
ruta_raiz = os.path.abspath(os.path.join(ruta_actual, '..'))
ruta_api_app_real = os.path.join(ruta_raiz, 'app')

# Aseguramos que la carpeta exista en la raíz
os.makedirs(ruta_api_app_real, exist_ok=True)

# 3. Creamos el camino completo hacia el archivo (SIN el .pkl al final)
camino_final_modelo = os.path.join(ruta_api_app_real, 'pipeline_ganador')

# 4. Guardamos con PyCaret en la ubicación correcta
save_model(modelo_final_lgb, camino_final_modelo)

print(f" ¡Modelo guardado con éxito!")
print(f" {camino_final_modelo}.pkl")

Transformation Pipeline and Model Successfully Saved
✅ ¡Modelo guardado con éxito en la ubicación REAL de tu API!
👉 C:\Users\ACER\Documents\Developer\Especializacion\Proyecto_3\app\pipeline_ganador.pkl
